<a href="https://colab.research.google.com/github/satyamkarn100-ctrl/FinDocAI/blob/main/02_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from torch.utils.data import Dataset
from torchvision import transforms
import os
from datasets import load_dataset
from PIL import Image


In [2]:
!pip install datasets -q

In [3]:
!wget -c "https://zenodo.org/records/10371464/files/FATURA2.zip?download=1" -O /content/FATURA2.zip

--2026-10-12 18:57:39--  https://zenodo.org/records/10371464/files/FATURA2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.184.98.114, 188.184.103.118, 188.185.48.75, ...
Connecting to zenodo.org (zenodo.org)|188.184.98.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 690717658 (659M) [application/octet-stream]
Saving to: ‘/content/FATURA2.zip’

/content/FATURA2.zi 100%[===================>] 658.72M  1.77MB/s    in 5m 5s   

2026-10-12 19:02:45 (2.16 MB/s) - ‘/content/FATURA2.zip’ saved [690717658/690717658]



In [4]:
!unzip -q /content/FATURA2.zip -d /content/FATURA

In [5]:
hf_data = load_dataset("Satyamkarn100/FinDoc-AI")

train_df = hf_data['train'].to_pandas()
val_df = hf_data['validation'].to_pandas()
test_df = hf_data['test'].to_pandas()

display(val_df.head())

README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

custom_train_split.csv:   0%|          | 0.00/362k [00:00<?, ?B/s]

custom_val_split.csv:   0%|          | 0.00/90.5k [00:00<?, ?B/s]

custom_test_split.csv:   0%|          | 0.00/111k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6400 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1600 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

,img_path,annot_path,template_id
0,Template3_Instance0.jpg,Template3_Instance0.json,3
1,Template3_Instance1.jpg,Template3_Instance1.json,3
2,Template3_Instance2.jpg,Template3_Instance2.json,3
3,Template3_Instance3.jpg,Template3_Instance3.json,3
4,Template3_Instance4.jpg,Template3_Instance4.json,3


In [6]:
class  FATURA_DATASET(Dataset):
  def __init__(self,dataframe,base_path,transform=None,target_transform=None):
    self.dataframe = dataframe
    self.base_path = base_path
    self.transform = transform
    self.target_transform=target_transform

  def __len__(self):
    return len(self.dataframe)

  def __getitem__(self,idx):
    image_name = self.dataframe.iloc[idx]['img_path']
    full_path = os.path.join(self.base_path,image_name)

    image = Image.open(full_path).convert("RGB")
    if self.transform:
      image = self.transform(image)

    if self.target_transform:
      image_name = self.target_transform(image_name)
    return image,image_name

In [9]:
from torch.utils.data import DataLoader

train_dataset = FATURA_DATASET(dataframe=train_df,base_path='/content/FATURA',transform=transforms.ToTensor())
val_dataset = FATURA_DATASET(dataframe=val_df,base_path='/content/FATURA',transform=transforms.ToTensor())
test_dataset = FATURA_DATASET(dataframe=test_df,base_path='/content/FATURA',transform=transforms.ToTensor())

train_dataloader = DataLoader(train_dataset,batch_size=64,shuffle=True)
val_dataloader = DataLoader(val_dataset,batch_size=64,shuffle=False)
test_dataloader = DataLoader(test_dataset,batch_size=64,shuffle=False)
